In [1]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from getpass import getpass

key = getpass("Paste your AT API key: ")
with open(".env", "w") as f:
    f.write(f"AT_API_KEY={key}\n")
with open(".gitignore", "a") as f:
    f.write(".env\n")
print("Done")

Paste your AT API key:  ········


Done


In [3]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
k = os.getenv("AT_API_KEY")
print("Key loaded:", k is not None, "| length:", len(k) if k else 0)

Key loaded: True | length: 32


In [4]:
import os, json, requests
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("AT_API_KEY")

url = "https://api.at.govt.nz/realtime/legacy/"
r = requests.get(url, headers={"Ocp-Apim-Subscription-Key": key}, timeout=30)
print("Status:", r.status_code)

if r.status_code != 200:
    print(r.text[:500])
else:
    data = r.json()
    print("Top-level keys:", list(data.keys()))
    resp = data.get("response", data)
    entities = resp.get("entity", [])
    print("Number of entities:", len(entities))
    for e in entities:
        if "trip_update" in e or "TripUpdate" in e:
            print(json.dumps(e, indent=2)[:2500])
            break
    else:
        if entities:
            print("First entity keys:", list(entities[0].keys()))
            print(json.dumps(entities[0], indent=2)[:2500])

Status: 200
Top-level keys: ['status', 'response', 'error']
Number of entities: 2248
{
  "id": "258-880002-76980-2-W191500-649367d3",
  "trip_update": {
    "trip": {
      "trip_id": "258-880002-76980-2-W191500-649367d3",
      "start_time": "21:23:00",
      "start_date": "20261103",
      "schedule_relationship": 0,
      "route_id": "E-W-201",
      "direction_id": 1
    },
    "stop_time_update": {
      "stop_sequence": 20,
      "arrival": {
        "delay": -65,
        "time": 1793697775,
        "uncertainty": 0
      },
      "departure": {
        "delay": -55,
        "time": 1793697785,
        "uncertainty": 4
      },
      "stop_id": "9405-8931d7d6",
      "schedule_relationship": 0
    },
    "vehicle": {
      "id": "59116",
      "label": "AMP        116",
      "license_plate": ""
    },
    "timestamp": 1793697775,
    "delay": -55
  },
  "is_deleted": false
}


In [5]:
%%writefile collector.py
import os, time, sqlite3, logging
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
KEY = os.getenv("AT_API_KEY")
URL = "https://api.at.govt.nz/realtime/legacy/"
DB = "bus_delays.db"
POLL_SECONDS = 60

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

SCHEMA = """
CREATE TABLE IF NOT EXISTS trip_delays (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    polled_at INTEGER,
    feed_ts INTEGER,
    trip_id TEXT,
    route_id TEXT,
    direction_id INTEGER,
    start_date TEXT,
    start_time TEXT,
    vehicle_id TEXT,
    stop_sequence INTEGER,
    stop_id TEXT,
    trip_delay INTEGER,
    stop_delay INTEGER,
    schedule_rel INTEGER,
    UNIQUE (trip_id, start_date, feed_ts)
);
CREATE INDEX IF NOT EXISTS idx_route_ts ON trip_delays (route_id, feed_ts);
"""

INSERT = """
INSERT OR IGNORE INTO trip_delays
(polled_at, feed_ts, trip_id, route_id, direction_id, start_date, start_time,
 vehicle_id, stop_sequence, stop_id, trip_delay, stop_delay, schedule_rel)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

def parse(entities, polled_at):
    rows = []
    for e in entities:
        tu = e.get("trip_update")
        if not tu:
            continue
        trip = tu.get("trip") or {}
        stu = tu.get("stop_time_update") or {}
        if isinstance(stu, list):
            stu = stu[0] if stu else {}
        ev = stu.get("arrival") or stu.get("departure") or {}
        veh = tu.get("vehicle") or {}
        rows.append((
            polled_at, tu.get("timestamp"), trip.get("trip_id"),
            trip.get("route_id"), trip.get("direction_id"),
            trip.get("start_date"), trip.get("start_time"),
            veh.get("id"), stu.get("stop_sequence"), stu.get("stop_id"),
            tu.get("delay"), ev.get("delay"), trip.get("schedule_relationship"),
        ))
    return rows

def main():
    con = sqlite3.connect(DB)
    con.executescript(SCHEMA)
    logging.info("Collector started. Press Ctrl+C to stop.")
    while True:
        start = time.time()
        try:
            r = requests.get(URL, headers={"Ocp-Apim-Subscription-Key": KEY}, timeout=30)
            r.raise_for_status()
            entities = (r.json().get("response") or {}).get("entity") or []
            rows = parse(entities, int(start))
            con.executemany(INSERT, rows)
            con.commit()
            logging.info("Saved %d trip updates", len(rows))
        except Exception as ex:
            logging.warning("Poll failed: %s", ex)
        time.sleep(max(1, POLL_SECONDS - (time.time() - start)))

if __name__ == "__main__":
    main()

Overwriting collector.py


In [6]:
import os
print(os.path.exists("collector.py"))

True


In [7]:
print(os.getcwd())

C:\Users\Aum Shukla\akl-bus-delays


In [8]:
import sqlite3, pandas as pd
con = sqlite3.connect("bus_delays.db")
print(pd.read_sql("SELECT COUNT(*) AS rows FROM trip_delays", con))
print(pd.read_sql("SELECT * FROM trip_delays LIMIT 5", con))

     rows
0  165010
   id   polled_at     feed_ts                                  trip_id  \
0   1  1792655691  1792655389  257-870008-69960-2-B172A177100-e5884c4c   
1   2  1792655691  1792655525  257-870005-72360-2-A174B185110-9be74534   
2   3  1792655691  1792652095  257-870006-66360-2-B162A167110-1677f651   
3   4  1792655691  1792599127  257-870056-78060-2-B182A987110-dbaaaed5   
4   5  1792655691  1792655396  257-870005-74160-2-A176B187130-154cb43b   

  route_id  direction_id start_date start_time vehicle_id  stop_sequence  \
0  S-C-201             1   20261022   19:26:00      59428             22   
1  S-C-201             0   20261022   20:06:00      59157             12   
2  S-C-201             1   20261022   18:26:00      59157             23   
3  S-C-201             1   20261022   21:41:00      59157              1   
4  S-C-201             0   20261022   20:36:00      59155              3   

         stop_id  trip_delay  stop_delay  schedule_rel  
0  9237-f5d0ce7f     

In [9]:
import sqlite3, pandas as pd
con = sqlite3.connect("bus_delays.db")
print(pd.read_sql("""
    SELECT COUNT(DISTINCT polled_at) AS polls,
           COUNT(DISTINCT route_id) AS routes,
           COUNT(DISTINCT trip_id) AS trips,
           datetime(MIN(polled_at),'unixepoch','+13 hours') AS first_poll_nz,
           datetime(MAX(polled_at),'unixepoch','+13 hours') AS last_poll_nz
    FROM trip_delays
""", con))

   polls  routes  trips        first_poll_nz         last_poll_nz
0    391     518  10623  2026-10-22 20:54:51  2026-10-28 16:24:25


In [10]:
import sqlite3, pandas as pd
con = sqlite3.connect("bus_delays.db")
polls = pd.read_sql("SELECT DISTINCT polled_at FROM trip_delays ORDER BY polled_at", con)
polls["nz_time"] = pd.to_datetime(polls["polled_at"], unit="s", utc=True).dt.tz_convert("Pacific/Auckland")
print(polls.groupby(polls["nz_time"].dt.floor("h")).size())
print("Largest gaps (minutes):")
print((polls["polled_at"].diff().sort_values(ascending=False).head(5) / 60).round(0))

nz_time
2026-10-22 20:00:00+13:00     3
2026-10-23 08:00:00+13:00    43
2026-10-23 09:00:00+13:00    60
2026-10-23 10:00:00+13:00    60
2026-10-23 11:00:00+13:00    60
2026-10-23 12:00:00+13:00    60
2026-10-23 13:00:00+13:00    60
2026-10-23 14:00:00+13:00    24
2026-10-28 16:00:00+13:00    21
dtype: int64
Largest gaps (minutes):
370    7301.0
3       681.0
1         1.0
258       1.0
267       1.0
Name: polled_at, dtype: float64


In [11]:
import requests, zipfile, io, os

r = requests.get("https://gtfs.at.govt.nz/gtfs.zip", timeout=120)
print(r.status_code, round(len(r.content) / 1e6, 1), "MB")
os.makedirs("gtfs", exist_ok=True)
zipfile.ZipFile(io.BytesIO(r.content)).extractall("gtfs")
print(os.listdir("gtfs"))

routes = pd.read_csv("gtfs/routes.txt")
print(routes["route_type"].value_counts())   # 3 = bus, 2 = rail, 4 = ferry
cal = pd.read_csv("gtfs/calendar.txt")
print("Service dates:", cal["start_date"].min(), "to", cal["end_date"].max())

200 24.6 MB
['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'fare_attributes.txt', 'fare_rules.txt', 'feed_info.txt', 'frequencies.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_times.txt', 'transfers.txt', 'trips.txt']
route_type
3    197
4     17
2      4
Name: count, dtype: int64
Service dates: 20261022 to 20270207


In [12]:
rt = pd.read_sql("SELECT DISTINCT route_id FROM trip_delays", con)
m = rt.merge(routes[["route_id", "route_short_name", "route_type"]], on="route_id", how="left")
print("Matched fraction:", round(m["route_short_name"].notna().mean(), 3))
print(m["route_type"].value_counts(dropna=False))

Matched fraction: 0.39
route_type
NaN    316
3.0    189
4.0     10
2.0      3
Name: count, dtype: int64


In [13]:
unmatched = m[m["route_short_name"].isna()]["route_id"]
print("Unmatched live route_ids:", unmatched.head(15).tolist())
print("Static route_ids:        ", routes["route_id"].head(15).tolist())

trips = pd.read_csv("gtfs/trips.txt", usecols=["trip_id", "route_id"])
rt_trips = pd.read_sql("SELECT DISTINCT trip_id FROM trip_delays", con)
print("Trip ID match fraction:", round(rt_trips["trip_id"].isin(trips["trip_id"]).mean(), 3))

cal = pd.read_csv("gtfs/calendar.txt")
print("Service dates:", cal["start_date"].min(), "to", cal["end_date"].max())

Unmatched live route_ids: ['S001-203', 'S001E-203', 'S001F-203', 'S001N-203', 'S002-218', 'S002A-203', 'S002D-203', 'S002F-203', 'S003-203', 'S003C-203', 'S003D-203', 'S004F-203', 'S005-218', 'S005B-203', 'S005C-203']
Static route_ids:         ['101-202', '186-203', '191-203', '105-202', '106-202', '195-203', '111-203', '112-203', '114-203', '392-203', '115-203', '116-203', '11T-207', '11W-207', '12-207']
Trip ID match fraction: 0.951
Service dates: 20261022 to 20270207


In [14]:
import numpy as np

trips = pd.read_csv("gtfs/trips.txt", usecols=["trip_id", "route_id", "trip_headsign"]).rename(columns={"route_id": "static_route_id"})
routes = pd.read_csv("gtfs/routes.txt", usecols=["route_id", "route_short_name", "route_type"]).rename(columns={"route_id": "static_route_id"})
tr = trips.merge(routes, on="static_route_id", how="left")

df = pd.read_sql("SELECT * FROM trip_delays", con)
print("Raw rows:", len(df))

df = df.merge(tr, on="trip_id", how="left")
print("Trip matched:", round(df["route_short_name"].notna().mean(), 3))

df = df[df["route_type"] == 3].copy()
print("Bus rows:", len(df), "| bus routes:", df["route_short_name"].nunique())

# How stale is each reading when we polled it?
df["age_s"] = df["polled_at"] - df["feed_ts"]
print(df["age_s"].describe(percentiles=[.5, .75, .9, .95]).round(0))
print("Delay (s):")
print(df["trip_delay"].describe(percentiles=[.01, .5, .99]).round(0))

Raw rows: 165010
Trip matched: 0.988
Bus rows: 156907 | bus routes: 189
count    156907.0
mean         58.0
std         274.0
min           0.0
50%          24.0
75%          39.0
90%          54.0
95%          61.0
max       11334.0
Name: age_s, dtype: float64
Delay (s):
count    156907.0
mean         23.0
std         250.0
min       -3245.0
1%         -696.0
50%          35.0
99%         657.0
max        9263.0
Name: trip_delay, dtype: float64


In [15]:
n0 = len(df)
clean = df[df["age_s"] <= 120]
print("After freshness filter:", len(clean), f"({len(clean)/n0:.1%})")
clean = clean[clean["schedule_rel"].fillna(0) == 0]          # drop cancelled trips
clean = clean[clean["trip_delay"].between(-900, 1800)].copy()
print("After delay clip:", len(clean), f"({len(clean)/n0:.1%})")
print("Exact-zero delays:", f"{(clean['trip_delay'] == 0).mean():.1%}")

clean["nz_time"] = pd.to_datetime(clean["feed_ts"], unit="s", utc=True).dt.tz_convert("Pacific/Auckland")
clean["hour"] = clean["nz_time"].dt.hour
clean["late5"] = (clean["trip_delay"] > 300).astype(int)    # more than 5 min late
clean["early1"] = (clean["trip_delay"] < -60).astype(int)   # more than 1 min early

by_hour = clean.groupby("hour").agg(
    readings=("trip_delay", "size"),
    median_delay_s=("trip_delay", "median"),
    pct_late5=("late5", "mean"),
    pct_early1=("early1", "mean"),
).round(3)
print(by_hour)

by_route = clean.groupby("route_short_name").agg(
    readings=("trip_delay", "size"),
    median_delay_s=("trip_delay", "median"),
    pct_late5=("late5", "mean"),
).query("readings >= 500").sort_values("pct_late5", ascending=False).round(3)
print(by_route.head(10))

After freshness filter: 153708 (98.0%)
After delay clip: 153173 (97.6%)
Exact-zero delays: 0.2%
      readings  median_delay_s  pct_late5  pct_early1
hour                                                 
8        21018             4.0      0.075       0.370
9        25001            17.0      0.061       0.326
10       21837            23.0      0.057       0.298
11       21786            42.0      0.094       0.265
12       21811            50.0      0.108       0.230
13       21804            48.0      0.100       0.247
14        8706            54.0      0.115       0.244
16       10409            58.0      0.177       0.277
20         801            42.0      0.086       0.232
                  readings  median_delay_s  pct_late5
route_short_name                                     
981                    518           197.0      0.388
712                    706           163.0      0.344
856                   1543           200.0      0.330
13                    1921           165

In [16]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = clean[["route_short_name", "direction_id", "hour", "stop_sequence", "trip_delay", "trip_id"]].dropna().copy()
d["route_code"] = d["route_short_name"].astype("category").cat.codes
feats = ["route_code", "direction_id", "hour", "stop_sequence"]

# Split by trip, so readings of the same trip never sit on both sides
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(d, groups=d["trip_id"]))
train, test = d.iloc[tr_idx], d.iloc[te_idx]

# Baseline 1: one global median for everything
global_med = train["trip_delay"].median()
b1 = np.full(len(test), global_med)

# Baseline 2: median delay per route
route_med = train.groupby("route_short_name")["trip_delay"].median()
b2 = test["route_short_name"].map(route_med).fillna(global_med)

# Model: gradient boosting with route as a categorical feature
model = HistGradientBoostingRegressor(
    categorical_features=[True, False, False, False],
    max_iter=200, learning_rate=0.08, random_state=42,
)
model.fit(train[feats], train["trip_delay"])
pred = model.predict(test[feats])

print("Train rows:", len(train), "| Test rows:", len(test))
print("MAE global median (s):", round(mean_absolute_error(test["trip_delay"], b1), 1))
print("MAE route median  (s):", round(mean_absolute_error(test["trip_delay"], b2), 1))
print("MAE gradient boost(s):", round(mean_absolute_error(test["trip_delay"], pred), 1))

Train rows: 115566 | Test rows: 37607
MAE global median (s): 168.7
MAE route median  (s): 160.6
MAE gradient boost(s): 137.1


In [17]:
from sklearn.inspection import permutation_importance

sample = test.sample(8000, random_state=42)
r = permutation_importance(model, sample[feats], sample["trip_delay"],
                           scoring="neg_mean_absolute_error", n_repeats=5, random_state=42)
for name, imp in sorted(zip(feats, r.importances_mean), key=lambda x: -x[1]):
    print(f"{name:15s} +{imp:6.1f} s worse when shuffled")

route_code      +  60.8 s worse when shuffled
stop_sequence   +  37.3 s worse when shuffled
direction_id    +  13.2 s worse when shuffled
hour            +  11.2 s worse when shuffled


In [18]:
import os, json, joblib

cats = list(d["route_short_name"].astype("category").cat.categories)
meta = {
    "route_names": cats,
    "median_stop_seq": d.groupby("route_short_name")["stop_sequence"].median().to_dict(),
    "hours_seen": sorted(int(h) for h in d["hour"].unique()),
    "trained_rows": int(len(d)),
    "mae_seconds": 125.4,
}
os.makedirs("api", exist_ok=True)
joblib.dump(model, "api/model.joblib")
json.dump(meta, open("api/meta.json", "w"))
print("Saved", len(cats), "routes; hours seen:", meta["hours_seen"])

Saved 188 routes; hours seen: [8, 9, 10, 11, 12, 13, 14, 16, 20]


In [19]:
from getpass import getpass

key = getpass("Paste your AT API key: ").strip()
with open(".env", "w") as f:
    f.write(f"AT_API_KEY={key}\n")
print("Saved. Length:", len(key))

Paste your AT API key:  ········


Saved. Length: 32


In [22]:
import gzip

n = 0
with gzip.open("snap.db.gz", "rb") as f:
    while chunk := f.read(1 << 20):
        n += len(chunk)
print("OK, uncompressed size:", round(n / 1e6), "MB")

OK, uncompressed size: 446 MB


In [7]:
import gzip, shutil, os

with gzip.open("snap.db.gz", "rb") as src, open("bus_delays_server.db", "wb") as dst:
    shutil.copyfileobj(src, dst)
print(round(os.path.getsize("bus_delays_server.db") / 1e6), "MB")

434 MB


In [24]:
with open(".gitignore", "a") as f:
    f.write("*.gz\n")

In [2]:
import os, sqlite3

for db in ["bus_delays.db", "bus_delays_server.db"]:
    if not os.path.exists(db):
        print(db, "| MISSING")
        continue
    with open(db, "rb") as f:
        head = f.read(16)
    print(db, "| size MB:", round(os.path.getsize(db) / 1e6, 1),
          "| valid SQLite header:", head == b"SQLite format 3\x00", "| first bytes:", head[:4])
    try:
        n = sqlite3.connect(db).execute("SELECT COUNT(*) FROM trip_delays").fetchone()[0]
        print("   rows:", n)
    except Exception as e:
        print("   cannot read:", e)

bus_delays.db | size MB: 30.4 | valid SQLite header: True | first bytes: b'SQLi'
   rows: 165010
bus_delays_server.db | size MB: 446.0 | valid SQLite header: False | first bytes: b'\x00\x00\x00\x00'
   cannot read: file is not a database


In [4]:
import gzip
with gzip.open("snap.db.gz", "rb") as f:
    head = f.read(16)
print(head, "| valid SQLite header:", head == b"SQLite format 3\x00")

b'SQLite format 3\x00' | valid SQLite header: True


In [8]:
import sqlite3, pandas as pd, numpy as np

COLS = "polled_at, feed_ts, trip_id, direction_id, start_date, stop_sequence, trip_delay"
WHERE = "polled_at - feed_ts <= 120 AND COALESCE(schedule_rel, 0) = 0 AND trip_delay BETWEEN -900 AND 1800"

def load(db):
    out = pd.read_sql(f"SELECT {COLS} FROM trip_delays WHERE {WHERE}", sqlite3.connect(db))
    out["source"] = db
    return out

raw = pd.concat([load("bus_delays.db"), load("bus_delays_server.db")], ignore_index=True)
raw = raw.drop_duplicates(subset=["trip_id", "start_date", "feed_ts"])
print("Rows after freshness and delay filters:", len(raw))

trips = pd.read_csv("gtfs/trips.txt", usecols=["trip_id", "route_id"], dtype=str).rename(columns={"route_id": "static_route_id"})
routes = pd.read_csv("gtfs/routes.txt", usecols=["route_id", "route_short_name", "route_type"],
                     dtype={"route_id": str, "route_short_name": str}).rename(columns={"route_id": "static_route_id"})
tr = trips.merge(routes, on="static_route_id", how="left")

df = raw.merge(tr, on="trip_id", how="left")
print("Trip match rate by source:")
print(df.groupby("source")["route_short_name"].apply(lambda s: round(s.notna().mean(), 3)))

df = df[df["route_type"] == 3].copy()
df["nz_time"] = pd.to_datetime(df["feed_ts"], unit="s", utc=True).dt.tz_convert("Pacific/Auckland")
df["hour"] = df["nz_time"].dt.hour
df["date"] = df["nz_time"].dt.date

print("Clean bus rows:", len(df))
print(df.groupby(["hour", "date"]).size().unstack("date").fillna(0).astype(int))

Rows after freshness and delay filters: 2616952
Trip match rate by source:
source
bus_delays.db           0.990
bus_delays_server.db    0.981
Name: route_short_name, dtype: float64
Clean bus rows: 2459667
date  2026-10-22  2026-10-23  2026-10-28  2026-10-29  2026-10-30  2026-10-31  \
hour                                                                           
0              0           0           0        2099        2047        2640   
1              0           0           0          60          63         536   
2              0           0           0           0           0         336   
3              0           0           0           0           0         356   
4              0           0           0         164         158          61   
5              0           0           0        5089        5132        1331   
6              0           0           0       17356       17245        9583   
7              0           0           0       27656       27005       1544

In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = df[["route_short_name", "direction_id", "hour", "stop_sequence", "trip_delay", "date"]].dropna().copy()
d["date"] = pd.to_datetime(d["date"])
d["day"] = d["date"].dt.strftime("%Y-%m-%d")
d["weekend"] = (d["date"].dt.dayofweek >= 5).astype(int)
d["route_code"] = d["route_short_name"].astype("category").cat.codes

F0 = ["route_code", "direction_id", "hour", "stop_sequence"]
F1 = F0 + ["weekend"]

def fit(train, feats):
    t = train.sample(min(len(train), 600_000), random_state=42)   # keeps each fit quick
    m = HistGradientBoostingRegressor(categorical_features=[True] + [False] * (len(feats) - 1),
                                      max_iter=200, learning_rate=0.08, random_state=42)
    return m.fit(t[feats], t["trip_delay"])

rows = []
for day in ["2026-10-31", "2026-11-01", "2026-11-02", "2026-11-03"]:
    train, test = d[d["day"] < day], d[d["day"] == day]
    y = test["trip_delay"]
    gmed = train["trip_delay"].median()
    rmed = train.groupby("route_short_name")["trip_delay"].median()
    rh = train.groupby(["route_short_name", "hour"])["trip_delay"].median()
    b_route = test["route_short_name"].map(rmed).fillna(gmed)
    keys = pd.MultiIndex.from_frame(test[["route_short_name", "hour"]])
    b_rh = pd.Series(rh.reindex(keys).values, index=test.index).fillna(b_route)
    m0, m1 = fit(train, F0), fit(train, F1)
    rows.append({"test_day": day, "test_rows": len(test),
                 "global": mean_absolute_error(y, np.full(len(test), gmed)),
                 "route": mean_absolute_error(y, b_route),
                 "route+hour": mean_absolute_error(y, b_rh),
                 "boost": mean_absolute_error(y, m0.predict(test[F0])),
                 "boost+weekend": mean_absolute_error(y, m1.predict(test[F1]))})
    print("done", day)

res = pd.DataFrame(rows).set_index("test_day").round(1)
print(res)
print("Mean over test days:")
print(res.drop(columns="test_rows").mean().round(1))

done 2026-10-31
done 2026-11-01
done 2026-11-02
done 2026-11-03
            test_rows  global  route  route+hour  boost  boost+weekend
test_day                                                              
2026-10-31     306620   159.4  153.1       158.3  145.5          145.5
2026-11-01     285265   158.2  151.1       154.6  145.0          149.2
2026-11-02     389653   171.6  163.5       162.1  148.9          147.1
2026-11-03     376134   168.0  158.7       155.4  137.4          135.8
Mean over test days:
global           164.3
route            156.6
route+hour       157.6
boost            144.2
boost+weekend    144.4
dtype: float64


In [10]:
import json, joblib

final = HistGradientBoostingRegressor(categorical_features=[True, False, False, False],
                                      max_iter=200, learning_rate=0.08, random_state=42)
final.fit(d[F0], d["trip_delay"])

cats = list(d["route_short_name"].astype("category").cat.categories)
counts = d["hour"].value_counts()
meta = {
    "route_names": cats,
    "median_stop_seq": d.groupby("route_short_name")["stop_sequence"].median().to_dict(),
    "hours_seen": sorted(int(h) for h, n in counts.items() if n >= 2000),
    "trained_rows": int(len(d)),
    "mae_seconds": 144.2,
}
joblib.dump(final, "api/model.joblib")
json.dump(meta, open("api/meta.json", "w"))
print("Saved:", len(cats), "routes |", meta["trained_rows"], "rows | hours recorded:", meta["hours_seen"])

Saved: 193 routes | 2459667 rows | hours recorded: [0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


In [11]:
import os, pathlib
from dotenv import load_dotenv

load_dotenv(override=True)
key = os.getenv("AT_API_KEY")
print("Key loaded:", bool(key) and len(key) > 30)

hits = []
for f in pathlib.Path(".").rglob("*"):
    if (f.is_file() and f.suffix in {".ipynb", ".py", ".md", ".txt", ".json", ".html"}
            and not {".git", "gtfs", "venv"} & set(f.parts) and f.name != ".env"):
        try:
            if key and key in f.read_text(errors="ignore"):
                hits.append(str(f))
        except Exception:
            pass
print("Files containing your key:", hits)

Key loaded: True
Files containing your key: []


In [12]:
from pathlib import Path

p = Path("api/index.html")
h = p.read_text(encoding="utf-8")

edits = [
 ('.sub{color:var(--muted);max-width:60ch;margin:0 0 24px}',
  '.sub{color:var(--muted);margin:0 0 24px;text-wrap:balance}'),
 ('Typical delays on Auckland buses, learned from live Auckland Transport data. Pick a route, then slide through the day.',
  'Typical delays on Auckland buses, learned from live Auckland Transport data.<br>Pick a route, then slide through the day.'),
]
for old, new in edits:
    assert h.count(old) == 1, "No unique match for: " + old[:60]
    h = h.replace(old, new)

p.write_text(h, encoding="utf-8")
print("Patched", len(edits), "edits")

Patched 2 edits


In [16]:
import os
print(os.getcwd())
print(os.path.exists(r"C:\Users\Aum Shukla\akl-bus-delays\docs"))

C:\Users\Aum Shukla\akl-bus-delays
True


In [24]:
from pathlib import Path

p = Path(r"C:\Users\Aum Shukla\akl-bus-delays\docs\README.md")
t = p.read_text(encoding="utf-8")

edits = [
 ("http://16.178.172.104/", "https://akl-bus.duckdns.org/"),
 (" (plain HTTP, running on a small AWS server, so it may be offline at times)", " (running on a small AWS server, so it may be offline at times)"),
 ("over plain HTTP, with no autoscaling or monitoring", "behind a Caddy reverse proxy on a free DuckDNS name, with no autoscaling or monitoring"),
 ("- Add HTTPS and basic monitoring.", "- Add basic monitoring."),
]
for old, new in edits:
    print(t.count(old), "x", old[:50])
    t = t.replace(old, new)

p.write_text(t, encoding="utf-8")

1 x http://16.178.172.104/
1 x  (plain HTTP, running on a small AWS server, so it
1 x over plain HTTP, with no autoscaling or monitoring
1 x - Add HTTPS and basic monitoring.


7404